# 08 — Telemetry-Triggered Guardrails and Protectability-Adjusted Risk

## Research Question
Can telemetry be used not only for measurement, but also for adaptive intervention and risk accounting?

## Hypothesis
Telemetry-triggered guardrails can approximate protectability: attacks that are observable are easier to mitigate than stealthy attacks.

## Input Data
- `episode_df_all`
- TDS summaries
- attack success proxy

## Prediction/Analysis Target
- Guardrail action, attack success proxy, telemetry stealthiness, protectability-adjusted risk

## Validation Protocol
Rule-based guardrail analysis and aggregate risk computation.

## Expected Paper Artifact
- Guardrail action distribution
- Attack success proxy
- Protectability-adjusted risk metric


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import silhouette_score
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    make_scorer,
)

from evaluation import (
    load_jsonl, 
    evaluate_binary_prediction, 
    evaluate_multiclass_attack_type,
    fit_rf_feature_importance,
    evaluate_multiclass_feature_group_ablation,
    add_akc_phase_from_attack_type,
    normalize_seed_column,
    infer_telemetry_features,
    prepare_akc_phase_df,
    run_akc_phase_leave_one_seed,
    summarize_multiclass_results,
    collect_akc_phase_predictions_leave_one_seed,
    plot_confusion_matrix_from_predictions,
    infer_early_akc_features,
    classification_report,
    run_temporal_akc_phase_classification,
    plot_temporal_akc_curves,
    adversarial_fragmentation_index,
    risk_conditioned_fragmentation,
    semantic_fragmentation_proxy,
    add_benign_normalized_fragmentation,
    evaluate_leave_one_seed_out,
    TELEMETRY_NUMERIC,
    evaluate_feature_group_ablation,
    FEATURE_GROUPS,
    evaluate_leave_one_attack_out,
    compute_permutation_importance,
    run_operational_only_leave_one_seed,
    summarize_results,
    plot_exp1_operational_comparison,
    load_tamas_jsonl,
    build_early_df_from_agent_outputs,
    run_early_detection_leave_one_seed,
    plot_early_detection_curve,
    EARLY_FEATURES_CANDIDATES,
    validate_early_df_for_attack_analysis,
    run_early_detection_by_attack_type,
    summarize_early_detection_by_attack,
    plot_metric_curves_by_attack,
    compute_temporal_gain,
    infer_numeric_features,
    keep_existing_numeric,
    run_early_detection_feature_ablation,
    plot_ablation_curves,
    telemetry_guardrail_decision,
    run_akc_phase_leave_one_attack_type_out,
)
from datasets.tamas.tamas_features import build_all_feature_tables

RESULTS_DIR = Path("results/tamas")
RAW_DIR = RESULTS_DIR / "raw"
FEATURES_DIR = RESULTS_DIR / "features"
PLOTS_DIR = RESULTS_DIR / "plots"

scenario = "education"
architecture = "centralized_tamas"
CONDITIONS = [
    "benign",
    "byzantine",
    "colluding",
    "contradicting",
    "DPI",
    "impersonation",
    "IPI",
]
SEEDS = [1, 2, 3]
MODEL_NAMES = [
    "ticlazau/meta-llama-3.1-8b-instruct:latest",
    # "qwen3",
]

In [ ]:

# Load processed telemetry tables generated by 00_setup_and_feature_extraction.ipynb.
# If files are not available yet, run notebook 00 first.
PROCESSED_DIR = Path("results/tamas/processed")

for _name in ["episode_df_all", "agent_df_all", "early_df_all", "df_raw_all"]:
    _path = PROCESSED_DIR / f"{_name}.parquet"
    if _path.exists():
        globals()[_name] = pd.read_parquet(_path)
        print(f"Loaded {_name}: {globals()[_name].shape}")
    else:
        print(f"Missing {_path}; run 00_setup_and_feature_extraction.ipynb if this notebook needs it.")


### Melhorar attack_success_proxy

Esse target está muito raro. Em vez de prever sucesso final, crie outcomes intermediários mais densos:

In [ ]:
# def add_behavioral_outcomes(df):
#     df = df.copy()

#     df["used_tool"] = (df["num_tool_calls"].fillna(0) > 0).astype(int)

#     df["high_tool_entropy"] = (
#         df["tool_call_entropy"].fillna(0)
#         > df["tool_call_entropy"].median()
#     ).astype(int)

#     df["high_contradiction"] = (
#         df["contradiction_marker_count"].fillna(0)
#         > df["contradiction_marker_count"].median()
#     ).astype(int)

#     df["high_latency"] = (
#         df["latency_total_s"].fillna(0)
#         > df["latency_total_s"].median()
#     ).astype(int)

#     df["high_token_cost"] = (
#         df["total_tokens"].fillna(0)
#         > df["total_tokens"].median()
#     ).astype(int)

#     df["high_fragmentation"] = (
#         df["risk_conditioned_fragmentation"].fillna(0)
#         > df["risk_conditioned_fragmentation"].median()
#     ).astype(int)

#     return df

# episode_fragmentation = add_behavioral_outcomes(episode_fragmentation)

# for target in [
#     "used_tool",
#     "high_tool_entropy",
#     "high_contradiction",
#     "high_latency",
#     "high_token_cost",
#     "high_fragmentation",
# ]:
#     print(target, episode_fragmentation[target].value_counts(normalize=True))

In [ ]:
episode_df_lst["guardrail_action"] = episode_df_lst.apply(
    telemetry_guardrail_decision,
    axis=1,
)

display(
    episode_df_lst.groupby(["attack_type", "guardrail_action"])
    .size()
    .reset_index(name="n")
)

In [ ]:
asp = episode_df_all[episode_df_all["seed"] == 1].copy()
asp = float(asp["attack_success_proxy"].sum()/len(asp["attack_success_proxy"]))
asp

In [ ]:
telemetry_stealthiness = 1 - tds["tds_bal_acc_mean"]
protectability_adjusted_risk = (asp * telemetry_stealthiness)
float(protectability_adjusted_risk[0])